# Day 067 | Regularization (Dropout, BatchNorm)
## 100 Days of ML for Geotechnical Engineering

| | |
|---|---|
| **Author** | Ripon Chandra Malo |
| **Phase** | 4 — Deep Learning |
| **Topic** | Regularization: L2, Dropout, BatchNorm, Early Stopping |
| **Tools** | PyTorch, NumPy, Matplotlib |
| **Geo Problem** | Preventing overfitting on small bearing capacity datasets (80 samples) |

---

## Concept Overview

| Technique | How It Works | Geotechnical Benefit |
|---|---|---|
| L2 Weight Decay | Adds ||w||² penalty to loss | Prevents large weights from fitting sensor noise |
| Dropout | Randomly zeros neurons during training | Forces ensemble-like robustness from sparse geo data |
| BatchNorm1d | Normalises activations per mini-batch | Stabilises training when features have different scales (kPa vs m) |
| Early Stopping | Halts training when val-loss stops improving | Saves the best model before overfitting begins |
| Overfitting | Low train loss, high val loss | Common with <100 geotechnical samples |

---

## Geotechnical Problem Framing

**Bearing capacity** from field tests is expensive — a typical project might yield only 80 data points from plate load tests and SPT borings. Neural networks overfit easily on such small datasets:

- A deep baseline model memorises training samples but fails on validation.
- Regularization techniques force the model to learn generalisable soil behaviour patterns.

**Features (4):** foundation width B (m), embedment depth Df (m), soil unit weight γ (kN/m³), friction angle φ (°)  
**Target:** ultimate bearing capacity qu (kPa)

---
## Part 1 | Setup & Synthetic Bearing Capacity Data

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 1 — Setup & Synthetic Bearing Capacity Dataset
# ─────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

# Synthetic bearing capacity data (Terzaghi-inspired, simplified)
N  = 80
B  = np.random.uniform(0.5, 3.0, N)     # foundation width (m)
Df = np.random.uniform(0.5, 2.5, N)     # embedment depth (m)
gamma = np.random.uniform(16, 21, N)    # unit weight (kN/m3)
phi   = np.random.uniform(20, 38, N)    # friction angle (degrees)

# Simplified bearing capacity formula + noise
Nc = 5.14 + 0.2 * phi
qu = gamma * Df * Nc + 0.5 * gamma * B * (phi / 30) + np.random.normal(0, 15, N)
qu = np.maximum(qu, 50)  # physical minimum

# Build tensors
X_raw = np.column_stack([B, Df, gamma, phi])
X_t   = torch.tensor(X_raw, dtype=torch.float32)
y_t   = torch.tensor(qu, dtype=torch.float32).view(-1, 1)

# Normalise
X_mean, X_std = X_t.mean(0), X_t.std(0)
y_mean, y_std = y_t.mean(),  y_t.std()
X_norm = (X_t - X_mean) / X_std
y_norm = (y_t - y_mean) / y_std

# Train / validation split (64 train, 16 val)
split     = 64
idx       = torch.randperm(N)
tr_idx, val_idx = idx[:split], idx[split:]

X_tr, y_tr   = X_norm[tr_idx], y_norm[tr_idx]
X_val, y_val = X_norm[val_idx], y_norm[val_idx]

tr_loader  = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=16)

print(f"Training samples: {len(X_tr)},  Validation samples: {len(X_val)}")
print(f"qu range: {qu.min():.0f} — {qu.max():.0f} kPa")

---
## Part 2 | Demonstrating Overfitting

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 2 — Baseline: Overfitting on Small Dataset
# ─────────────────────────────────────────────────────────

class BaselineNet(nn.Module):
    """Deliberately deep network — will overfit on 64 samples."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x)

def evaluate(model, loader, criterion):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            total += criterion(model(xb), yb).item() * len(xb)
    return total / len(loader.dataset)

def train_full(model, optimizer, epochs=300):
    criterion = nn.MSELoss()
    tr_hist, val_hist = [], []
    for _ in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        tr_hist.append(evaluate(model, tr_loader, criterion))
        val_hist.append(evaluate(model, val_loader, criterion))
    return tr_hist, val_hist

torch.manual_seed(0)
baseline = BaselineNet()
tr_hist, val_hist = train_full(baseline, torch.optim.Adam(baseline.parameters(), lr=1e-3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tr_hist,  label='Train Loss',      lw=2)
ax.plot(val_hist, label='Validation Loss', lw=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSELoss (normalised)')
ax.set_title('Overfitting Demo — Deep Baseline on 64 Bearing Capacity Samples')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gap = val_hist[-1] - tr_hist[-1]
print(f"Final train loss: {tr_hist[-1]:.4f}")
print(f"Final val  loss:  {val_hist[-1]:.4f}")
print(f"Overfitting gap:  {gap:.4f}  ← target to reduce with regularization")

---
## Part 3 | L2 Weight Decay

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 3 — L2 Weight Decay (via AdamW weight_decay)
# ─────────────────────────────────────────────────────────

torch.manual_seed(0)
model_l2 = BaselineNet()
opt_l2   = torch.optim.AdamW(model_l2.parameters(), lr=1e-3, weight_decay=1e-2)
tr_l2, val_l2 = train_full(model_l2, opt_l2)

print(f"L2 regularised — final val loss: {val_l2[-1]:.4f}  (baseline: {val_hist[-1]:.4f})")

# Show weight magnitude reduction
baseline_w_norm = sum(p.norm().item() for p in baseline.parameters())
l2_w_norm       = sum(p.norm().item() for p in model_l2.parameters())
print(f"\nWeight L2-norm — Baseline: {baseline_w_norm:.2f},  L2-regularised: {l2_w_norm:.2f}")
print("L2 penalty keeps weights small => smoother decision boundaries")

---
## Part 4 | Dropout

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 4 — Dropout Regularization
# ─────────────────────────────────────────────────────────

class DropoutNet(nn.Module):
    """Same depth as baseline but with Dropout(0.3) after each hidden layer."""
    def __init__(self, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x)

torch.manual_seed(0)
model_do = DropoutNet(p=0.3)
opt_do   = torch.optim.Adam(model_do.parameters(), lr=1e-3)
tr_do, val_do = train_full(model_do, opt_do)

print(f"Dropout(0.3)  — final val loss: {val_do[-1]:.4f}  (baseline: {val_hist[-1]:.4f})")
print()
print("Dropout concept for geo engineers:")
print("  Randomly silencing neurons prevents co-adaptation — like testing")
print("  your soil model without one boring at a time, forcing robustness.")

---
## Part 5 | Batch Normalization

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 5 — BatchNorm1d
# ─────────────────────────────────────────────────────────

class BatchNormNet(nn.Module):
    """BatchNorm1d after each hidden layer (before activation)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x)

torch.manual_seed(0)
model_bn = BatchNormNet()
opt_bn   = torch.optim.Adam(model_bn.parameters(), lr=1e-3)
tr_bn, val_bn = train_full(model_bn, opt_bn)

print(f"BatchNorm1d   — final val loss: {val_bn[-1]:.4f}  (baseline: {val_hist[-1]:.4f})")
print()
print("BatchNorm benefit for geo data:")
print("  Bearing capacity inputs span very different scales:")
print("    B: 0.5—3.0 m | gamma: 16—21 kN/m3 | phi: 20—38°")
print("  BatchNorm normalises internal activations to mean~0, std~1")
print("  => faster training and reduced sensitivity to input scale differences.")

---
## Part 6 | Early Stopping Implementation

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 6 — Early Stopping
# ─────────────────────────────────────────────────────────

class EarlyStopper:
    """Stops training when validation loss doesn't improve for `patience` epochs."""
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience   # True => stop

torch.manual_seed(0)
model_es = BaselineNet()
opt_es   = torch.optim.Adam(model_es.parameters(), lr=1e-3)
criterion = nn.MSELoss()
stopper   = EarlyStopper(patience=25)

tr_es, val_es = [], []
stopped_at = 300
for epoch in range(300):
    model_es.train()
    for xb, yb in tr_loader:
        opt_es.zero_grad()
        criterion(model_es(xb), yb).backward()
        opt_es.step()
    tl = evaluate(model_es, tr_loader,  criterion)
    vl = evaluate(model_es, val_loader, criterion)
    tr_es.append(tl); val_es.append(vl)
    if stopper.step(vl, model_es):
        stopped_at = epoch + 1
        print(f"Early stopping at epoch {stopped_at}  (best val loss: {stopper.best_loss:.4f})")
        break

# Restore best weights
model_es.load_state_dict(stopper.best_state)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tr_es,  label='Train',      lw=2)
ax.plot(val_es, label='Validation', lw=2, linestyle='--')
ax.axvline(stopped_at - 1, color='red', linestyle=':', label=f'Stopped at {stopped_at}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSELoss')
ax.set_title('Early Stopping — Baseline Net on Bearing Capacity')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Part 7 | Comparing All Four Models

In [ ]:
# ─────────────────────────────────────────────────────────
# PART 7 — Side-by-Side Comparison: 4 Regularization Strategies
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=False)
configs = [
    ('Baseline (no regularization)', tr_hist, val_hist),
    ('L2 Weight Decay (AdamW)',       tr_l2,   val_l2),
    ('Dropout (p=0.3)',               tr_do,   val_do),
    ('BatchNorm1d',                   tr_bn,   val_bn),
]

for ax, (title, tr, val) in zip(axes.flat, configs):
    ax.plot(tr,  label='Train',      lw=2)
    ax.plot(val, label='Validation', lw=2, linestyle='--')
    gap = val[-1] - tr[-1]
    ax.set_title(f'{title}\nFinal gap: {gap:.4f}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSELoss')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Regularization Comparison — Bearing Capacity Regression (80 samples)', y=1.01)
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"{'Method':25s} | Train Loss | Val Loss | Gap")
print("-" * 60)
for title, tr, val in configs:
    print(f"{title:25s} | {tr[-1]:.4f}     | {val[-1]:.4f}   | {val[-1]-tr[-1]:.4f}")

---
## Practice Exercises

1. **Dropout tuning** — Try `p=0.1`, `p=0.3`, `p=0.5`. What happens to both train and val loss as dropout increases?
2. **Combined regularization** — Build a network using both Dropout AND BatchNorm AND L2 weight decay. Does combining all three improve val loss further?
3. **Early stopping patience** — Reduce patience to 10. How many epochs does training run? Increase to 50. What is the tradeoff?
4. **Feature scaling without BatchNorm** — Remove the manual normalisation of X (use raw B, Df, gamma, phi). Does BatchNorm still help more than Dropout?
5. **Dataset size sensitivity** — Repeat the comparison with N=40 (extreme small data) and N=200. At which size does overfitting become less of a problem?

In [ ]:
# ─────────────────────────────────────────────────────────
# PRACTICE — Starter code
# ─────────────────────────────────────────────────────────

# Exercise 2: Combined regularization
# class CombinedNet(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(4, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(128, 1)
#         )
#     def forward(self, x): return self.net(x)
# model_combined = CombinedNet()
# opt = torch.optim.AdamW(model_combined.parameters(), lr=1e-3, weight_decay=1e-2)
# ...

print("Practice exercise starters are ready — uncomment and complete!")

---
## ML Connection Table

| Geotechnical Scenario | Recommended Regularization | Reason |
|---|---|---|
| 50–100 bearing capacity samples | Dropout(0.3) + Early Stopping | Small N; prevent memorisation |
| Mixed-scale features (kPa, m, °) | BatchNorm1d | Internal activation normalisation |
| Settlement data with sensor drift | L2 weight decay | Keeps model smooth despite noise |
| CPT classification (1000+ profiles) | Minimal regularization needed | Enough data to learn naturally |
| Embankment stability transfer | Dropout + L2 | Prevents overfitting to site-specific patterns |

---
## Key Takeaways

1. **Overfitting is the default outcome** on small geotechnical datasets — a deep network will memorise 64 bearing capacity samples rather than learn soil mechanics.
2. **L2 weight decay** (via AdamW `weight_decay`) suppresses large parameter values, producing smoother predictions that generalise better to unseen site conditions.
3. **Dropout** randomly disables neurons during training, forcing the network to distribute knowledge across multiple paths — analogous to not relying on a single soil boring.
4. **BatchNorm1d** normalises internal activations per mini-batch, making training stable when input features span different physical scales (kPa vs. meters vs. degrees).
5. **Early stopping** is free regularization — always monitor validation loss and restore the best checkpoint to avoid the overfitting region.

---
*Day 067 of 100 — Regularization is not about limiting your model — it is about teaching it to not over-trust its training data.*